# Data Quality Issues

Dataset: Retail Store Sales (dirty), 12,575 rows × 11 columns

1. Discount Applied: 4,199 missing (33%)
2. Item: 1,213 missing (~10%); may be deducible from Category + Price Per Unit
3. Price Per Unit / Quantity / Total Spent: ~600 missing each; recoverable via
   Total Spent = Quantity × Price Per Unit when two of three are present
4. Transaction Date: stored as string, needs conversion to datetime
5. Discount Applied: stored as object, should be boolean/category

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/retail_store_sales.csv")
print(df.shape)
df.info()

(12575, 11)
<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.9+ MB


In [3]:
df.head(10)                                   # eyeball actual values

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
5,TXN_7482416,CUST_09,Patisserie,NaN,NaN,10.0,200.0,Credit Card,Online,2023-11-30,NaN
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True
8,TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,False
9,TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,False


In [4]:
df.isna().sum()                               # exact missing counts per column

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [5]:
df.duplicated().sum()                         # any fully duplicated rows?

np.int64(0)

In [6]:
for col in ["Category", "Payment Method", "Location", "Discount Applied"]:
    print(col, "→", df[col].unique(), "\n")   # inconsistent spellings? weird values?

Category → <ArrowStringArray>
[                        'Patisserie',                      'Milk Products',
                           'Butchers',                          'Beverages',
                               'Food',                          'Furniture',
      'Electric household essentials', 'Computers and electric accessories']
Length: 8, dtype: str 

Payment Method → <ArrowStringArray>
['Digital Wallet', 'Credit Card', 'Cash']
Length: 3, dtype: str 

Location → <ArrowStringArray>
['Online', 'In-store']
Length: 2, dtype: str 

Discount Applied → [True False nan] 



In [7]:
df.describe()                                 # numeric ranges — negative prices? crazy outliers?

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [8]:
df.duplicated().sum()   # what did this return earlier?

np.int64(0)

In [9]:
# does the golden equation actually hold where everything is present?
complete = df.dropna(subset=["Price Per Unit", "Quantity", "Total Spent"])
mismatch = complete[abs(complete["Total Spent"] - complete["Quantity"] * complete["Price Per Unit"]) > 0.01]
print(len(mismatch), "rows where Total ≠ Qty × Price")

0 rows where Total ≠ Qty × Price


In [10]:
# can Items really be deduced? check if Category + Price uniquely identifies an Item
pairs = df.dropna(subset=["Item"]).groupby(["Category", "Price Per Unit"])["Item"].nunique()
print((pairs > 1).sum(), "ambiguous Category+Price combos out of", len(pairs))

0 ambiguous Category+Price combos out of 200


In [11]:
import sys, logging
sys.path.append("..")          # lets the notebook see the src folder
logging.basicConfig(level=logging.INFO)

from src.cleaning import recover_missing_numerics

df_clean = recover_missing_numerics(df)
print("Before:", df[["Price Per Unit", "Quantity", "Total Spent"]].isna().sum().sum(), "missing")
print("After: ", df_clean[["Price Per Unit", "Quantity", "Total Spent"]].isna().sum().sum(), "missing")

INFO:src.cleaning:Recovered 609 missing values in Price Per Unit
INFO:src.cleaning:Recovered 0 missing values in Quantity
INFO:src.cleaning:Recovered 0 missing values in Total Spent


Before: 1817 missing
After:  1208 missing


In [12]:
both = (df["Quantity"].isna() & df["Total Spent"].isna()).sum()
print("Rows missing BOTH Quantity and Total:", both)
print("Rows missing only Quantity:", (df["Quantity"].isna() & df["Total Spent"].notna()).sum())
print("Rows missing only Total:  ", (df["Total Spent"].isna() & df["Quantity"].notna()).sum())

Rows missing BOTH Quantity and Total: 604
Rows missing only Quantity: 0
Rows missing only Total:   0


In [13]:
from src.cleaning import recover_items

df_clean2 = recover_items(df_clean)
print("Items still missing:", df_clean2["Item"].isna().sum(), "of originally", df["Item"].isna().sum())

INFO:src.cleaning:Recovered 1213 missing values in Item


Items still missing: 0 of originally 1213


In [14]:
from src.cleaning import convert_dates, clean_discount

df_clean3 = convert_dates(df_clean2)
df_clean4 = clean_discount(df_clean3)

INFO:src.cleaning:Converted Transaction Date to datetime (0 unparseable)
INFO:src.cleaning:Converted Discount Applied to nullable boolean (4199 NA kept)


In [15]:
from src.cleaning import clean

df_final = clean(pd.read_csv("../data/raw/retail_store_sales.csv"))
df_final.to_parquet("../data/processed/retail_store_sales_clean.parquet")
df_final.info()

INFO:src.cleaning:Starting cleaning pipeline: 12575 rows, 11 columns
INFO:src.cleaning:Recovered 609 missing values in Price Per Unit
INFO:src.cleaning:Recovered 0 missing values in Quantity
INFO:src.cleaning:Recovered 0 missing values in Total Spent
INFO:src.cleaning:Recovered 1213 missing values in Item
INFO:src.cleaning:Converted Transaction Date to datetime (0 unparseable)
INFO:src.cleaning:Converted Discount Applied to nullable boolean (4199 NA kept)
INFO:src.cleaning:Pipeline finished: 5407 values still missing


<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  str           
 1   Customer ID       12575 non-null  str           
 2   Category          12575 non-null  str           
 3   Item              12575 non-null  str           
 4   Price Per Unit    12575 non-null  float64       
 5   Quantity          11971 non-null  float64       
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    12575 non-null  str           
 8   Location          12575 non-null  str           
 9   Transaction Date  12575 non-null  datetime64[us]
 10  Discount Applied  8376 non-null   boolean       
dtypes: boolean(1), datetime64[us](1), float64(3), str(6)
memory usage: 1.7 MB
